# MobileNet V2 Model

In [3]:
# 1. Install Kaggle CLI
!pip install -q kaggle

In [4]:
!kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset

Dataset URL: https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset
License(s): Attribution 4.0 International (CC BY 4.0)
100% 157M/157M [00:00<00:00, 204MB/s] 



In [ ]:
# Unzip the images
!unzip -qn brain-tumor-mri-dataset.zip -d brain_tumor_data

In [13]:
# Check the directory structure
print("Training data:")
!cd brain_tumor_data && cd Training && ls
print("\nTesting data:")
!cd brain_tumor_data && cd Testing && ls

Training data:
glioma	meningioma  notumor  pituitary

Testing data:
glioma	meningioma  notumor  pituitary


In [15]:
import getpass

github_token = getpass.getpass("Enter github token to clone the repository: ")

# Cloning the repo
!git clone https://{github_token}@github.com/Dini-s/brain-tumour-mri-classification.git

Cloning into 'brain-tumour-mri-classification'...
remote: Enumerating objects: 131, done.
remote: Counting objects: 100% (131/131), done.
remote: Compressing objects: 100% (110/110), done.
remote: Total 131 (delta 63), reused 69 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (131/131), 4.61 MiB | 17.81 MiB/s, done.
Resolving deltas: 100% (63/63), done.


In [44]:
# add the path to the cloned repo to sys.path
import sys
sys.path.append('brain-tumour-mri-classification')

#importing the modules within src folder
from src.dataset import get_dataloaders
from src.utils import confusion_matrix, print_metrics, plot_confusion_matrix, plot_learning_curves, evaluate_model
from src.transforms import train_transforms, val_test_transforms

#importing the required libraries
import torch
import torch.nn as nn
import torch.optim as optimizations
import torchvision.models as models
import time
from pathlib import Path

# <h3> Settings for the whole notebook <h3>

In [45]:
DATA_DIR = '/content/brain_tumor_data'
BATCH_SIZE = 32
SEED = 42
NUM_CLASSES = 4
CHECKPOINT_DIR = Path('/content/mobilenetv2_best.pth')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# <h3> Loading Data <h3>

In [82]:
training, validation, test = get_dataloaders(DATA_DIR, BATCH_SIZE)

image, label = next(iter(training))

print(f"Image shape: {image.shape}")
print(f"Label: {label}")

print(f"Number of images in the batch: {len(image)}")

Train:      4,480 images
Validation: 1,120 images
Test:       1,600 images
Classes:    ['glioma', 'meningioma', 'notumor', 'pituitary']
Image shape: torch.Size([32, 3, 224, 224])
Label: tensor([1, 1, 3, 1, 1, 3, 3, 0, 0, 0, 1, 2, 0, 1, 1, 0, 0, 2, 2, 0, 3, 0, 1, 3,
        0, 2, 3, 1, 0, 3, 1, 3])
Number of images in the batch: 32


# <h3>Building the Model</h3>

importing the model and replacing the classifier layer with a new layer. The new layer has 4 output classes

In [104]:
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

model = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT, progress=True)

classifier = model.classifier
input_features = classifier[1].in_features

# setting the manula seed
torch.manual_seed(SEED)

# new layer with 4 output classes
classifier[1] = nn.Linear(in_features=input_features, out_features=NUM_CLASSES, bias=True)

print(classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=False)
  (1): Linear(in_features=1280, out_features=4, bias=True)
)


counting the number of trainable parameters in the model

In [ ]:
global COUNT

def count_parameters(model):
    COUNT = 0
    for parameters in model.parameters():
        if parameters.requires_grad:
            COUNT += parameters.numel()

    print(f"Total number of trainable parameters: {COUNT}")

count_parameters(model)

Total number of trainable parameters: 2228996


<b>Running the model on a batch of images in evaluation mode and without gradients</b>

In [157]:
# Moving the model and data to device
moved_model = model.to(DEVICE)
moved_images_batch = image.to(DEVICE)
moved_labels = label.to(DEVICE)
# 1 test forward pass
with torch.no_grad():
    moved_model.eval()
    output = moved_model(moved_images_batch)

<b>Calculating the loss</b>

In [158]:
loss_function = nn.CrossEntropyLoss()
loss = loss_function(output, moved_labels)
print(f"Loss: {loss.item()}")

Loss: 1.3548181056976318
